# Теомор — бесшовная текстура (Colab / Py3.13)

Если сессия уже красная: **Среда выполнения → Перезапустить сеанс → GPU**, потом ▶ сверху вниз.


## 1) GPU


In [ ]:
import sys, torch
print("python", sys.version.split()[0])
print("cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Включи GPU: Среда выполнения → Сменить среду → GPU")
print("gpu", torch.cuda.get_device_name(0))


## 2) Пакеты + патч Pillow._Ink


In [ ]:
# setuptools новый ломает torch на Colab — чуть откатим
%pip -q install -U pip wheel
%pip -q install "setuptools>=70,<82"
%pip -q install --only-binary=:all: "tokenizers>=0.20.3"
%pip -q install -U "pillow>=11.0.0" "diffusers>=0.32.0" "transformers>=4.46.0,<5" "accelerate" "safetensors" "huggingface_hub"

# Pillow 11 убрал PIL.typing._Ink — diffusers/старые куски ещё импортируют.
# Патч ДО import diffusers:
import typing
import PIL.typing as _pil_typing
if not hasattr(_pil_typing, "_Ink"):
    _pil_typing._Ink = typing.Any
    print("patched PIL.typing._Ink")

from PIL import Image, ImageDraw
import diffusers, transformers, tokenizers
print("PIL", __import__("PIL").__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)
print("tokenizers", tokenizers.__version__)
print("ImageDraw OK")


## 3) Модель


In [ ]:
import typing
import PIL.typing as _pil_typing
if not hasattr(_pil_typing, "_Ink"):
    _pil_typing._Ink = typing.Any

import torch
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageDraw

pipe = StableDiffusionPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
print("model ready")


## 4) Промпт


In [ ]:
ANCHOR = (
    "Teomor style: dirty parchment yellow, diseased purple, black ink strokes, "
    "one dominant color, PS2 Silent Hill + Dread Delusion grit, hand-painted look, "
    "halftone grain, desaturated, never clean PBR, never neon, never glossy, "
    "seamless tileable texture, flat albedo, no perspective, no characters, no UI"
)
NEGATIVE = (
    "clean PBR, photoreal, sharp 8k, neon, anime, plastic, glossy, chrome, bloom, "
    "depth of field, text, logo, border, character, face, perspective room"
)
SLOT = (
    "seamless tileable damp alley wall albedo, diseased muddy purple dominant, "
    "wet dirt, mushroom spore stains, black ink puddles, sickly paper in cracks, "
    "Silent Hill fog grit, hand painted, not photo cobblestone"
)
prompt = ANCHOR + ". " + SLOT
print(prompt[:180], "...")


## 5) Генерация


In [ ]:
import random
seed = random.randint(1, 999999)
generator = torch.Generator("cuda").manual_seed(seed)
print("seed", seed)
result = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE,
    width=512,
    height=512,
    num_inference_steps=28,
    guidance_scale=6.5,
    generator=generator,
)
img = result.images[0]
img.save("/content/teomor_tile_raw.png")
display(img)
print("saved /content/teomor_tile_raw.png")


## 6) Seamless


In [ ]:
import numpy as np
from PIL import Image

def make_seamless(im: Image.Image, blend: int = 56) -> Image.Image:
    a = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
    h, w, _ = a.shape
    b = min(blend, h // 4, w // 4)
    out = a.copy()
    for i in range(b):
        t = i / max(1, b - 1)
        wgt = 0.5 - 0.5 * np.cos(np.pi * t)
        out[:, i] = a[:, i] * (1 - wgt) + a[:, w - 1 - i] * wgt
        out[:, w - 1 - i] = a[:, w - 1 - i] * (1 - wgt) + a[:, i] * wgt
        out[i] = out[i] * (1 - wgt) + out[h - 1 - i] * wgt
        out[h - 1 - i] = out[h - 1 - i] * (1 - wgt) + out[i] * wgt
    return Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8))

raw = Image.open("/content/teomor_tile_raw.png")
seamless = make_seamless(raw)
seamless.save("/content/teomor_tile_seamless.png")
display(seamless)
print("Files → teomor_tile_seamless.png → Download")
